# Qwen3 VL 4B 단일선 추적 활성값 수집

고정한 10문제 중 **첫 1문제만 실행한 뒤 중지**합니다. T4 런타임을 선택하고 순서대로 실행하세요. ViT·merger·DeepStack·decoder의 전 block 활성값과 모든 head의 질문/답 예측/생성 위치→이미지 attention을 저장합니다. 실제 데이터 크기와 최대 VRAM을 확인한 다음 맨 아래의 재개 cell을 사용합니다.


In [ ]:
from pathlib import Path
import importlib.metadata
import os
import subprocess
import sys

REPO = Path('/content/mllm-line-tracing')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/seungjoolee24/mllm-line-tracing.git', str(REPO)], check=True)
os.chdir(REPO)
EXP = REPO / 'experiments/exp001_qwen3vl4b_trace10'
assert (EXP / 'collect.py').is_file(), 'Clone does not contain the experiment. Use the current published notebook/source.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(EXP / 'requirements-colab.txt')], check=True)
if not importlib.metadata.version('torch').startswith('2.8.0') or not importlib.metadata.version('torchvision').startswith('0.23.0'):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--index-url',
                    'https://download.pytorch.org/whl/cu126', 'torch==2.8.0', 'torchvision==0.23.0'], check=True)
print('Setup complete. Model execution happens in a separate Python process.')


## T4와 고정 설정 확인

설정이 T4가 아니면 수집을 시작하지 않습니다. 결과는 이 실험 폴더의 results/run001 아래에 저장됩니다.


In [ ]:
import json
CONFIG = json.loads((EXP / 'config.json').read_text())
RUN = EXP / 'results/run001'
DATASET = EXP / 'dataset'
RUN.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
source_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
(RUN / 'notebook_provenance.json').write_text(json.dumps({'source_commit': source_sha, 'notebook': 'activation_pilot.ipynb'}, indent=2))
subprocess.run([sys.executable, str(EXP / 'prepare_cases.py')], check=True)
print(json.dumps(CONFIG, indent=2))


In [ ]:
from PIL import Image
from IPython.display import display
with (DATASET / 'annotations.jsonl').open() as stream:
    first_case = json.loads(next(stream))
display(Image.open(DATASET / first_case['files']['image']))
print(first_case['id'], first_case['prompt'])


## 첫 1문제 수집

자유 생성 후 생성된 답을 causal replay합니다. 첫 생성 token의 예측 위치를 마지막 prompt 위치에 대응시키며 정렬 검증이 실패하면 정상 완료로 기록하지 않습니다. 기존 완료 파일은 hash·shape·dtype를 검증한 뒤 재사용합니다.


In [ ]:
BASE_COMMAND = [sys.executable, str(EXP / 'collect.py'), '--dataset', str(DATASET), '--output', str(RUN),
                '--model-id', CONFIG['model_id'], '--model-revision', CONFIG['model_revision'],
                '--max-new-tokens', str(CONFIG['max_new_tokens']), '--require-t4', '--resume']
subprocess.run(BASE_COMMAND + ['--limit', '1'], check=True)


## 용량 보고와 결과 패키징

원시 활성값은 layer별 .pt이며 각 파일의 shape·dtype·SHA256을 meta.json에 기록합니다. 아래에서 원시 파일 크기와 ZIP 크기를 함께 확인합니다. 10문제 예상치는 첫 문제의 단순 선형 환산이며 실제 생성 길이에 따라 달라집니다.


In [ ]:
subprocess.run([sys.executable, str(EXP / 'summarize.py'), '--run', str(RUN), '--dataset', str(DATASET), '--archive'], check=True)
report = json.loads((RUN / 'size_report.json').read_text())
print(f"완료 {report['completed_cases']}/{report['intended_cases']}, tensor {report['tensor_file_mib']:.1f} MiB")
print(json.dumps(report, indent=2))


## 첫 결과 다운로드

Colab 런타임 저장소는 임시입니다. 아래 ZIP에는 첫 문제의 원시 텐서와 metadata가 모두 들어 있습니다. 재개 시에는 동일 RUN 폴더를 유지하거나 이 ZIP을 같은 위치에 풀어 둡니다.


In [ ]:
from google.colab import files
ARCHIVE = RUN.parent / (RUN.name + '.zip')
files.download(str(ARCHIVE))


## 나머지 9문제 재개

이번에는 실행하지 않습니다. 첫 결과 용량을 확인한 뒤 CONTINUE_REMAINING을 True로 바꾸고 이 cell만 실행하면 동일 설정으로 완료된 문제를 검증하고 나머지를 수집합니다.


In [ ]:
CONTINUE_REMAINING = False
if CONTINUE_REMAINING:
    subprocess.run(BASE_COMMAND, check=True)
    subprocess.run([sys.executable, str(EXP / 'summarize.py'), '--run', str(RUN), '--dataset', str(DATASET), '--archive'], check=True)
    files.download(str(RUN.parent / (RUN.name + '.zip')))
else:
    print('추가 수집 중지. 첫 문제의 실제 용량을 검토한 후 재개하세요.')
